# Notebook 01 — Spatial Health Data: Structure, Location, and What It Tells Us
**GeoAI for Climate, Environment and Health**

---

> **Where you are in the pipeline**
>
> ```
> [📋 Health records] → 🛰 Satellite → 🌍 Environment → 🔗 Link → ⏱ Timing → 🎯 Results & GeoAI
> ```
>
> This is your foundation notebook. Every GeoAI analysis in this course starts here:
> with people, places, and outcomes. Before we can ask what the environment does to health,
> we need a dataset of individuals — each with a **location**, a **time**, and a **health outcome**.
> We use the Zimbabwe Demographic and Health Survey (DHS) 2015 birth records.

**By the end of this notebook you should be able to:**
- Describe the common attributes of a spatial health dataset
- Explain why the choice of spatial unit (cluster, district, facility) shapes what you can and cannot infer
- Identify the key quality issues in DHS birth data: missing values, sentinel codes, and coordinate displacement
- Observe urban–rural gradients in birth outcomes and explain why location matters

---

---
## 1. Types of Spatial Health Data — and Why the Unit Matters

Health research uses several types of georeferenced data. The **spatial unit** you work with shapes every downstream analysis: what exposures can be linked, what biases arise, and what inferences are valid.

| Spatial unit | Typical source | Geographic reference | Best suited for |
|---|---|---|---|
| **Health facility** | MoH registry, HMIS | GPS coordinate of the clinic/hospital | Access analysis, catchment delineation, service-quality studies |
| **DHS cluster** | DHS Programme household survey | Cluster centroid, displaced 0–5 km for privacy | Individual-level outcomes from a probability sample |
| **Administrative area** | National census, national GIS | District or ward polygon | Population surveillance, routine aggregate reporting |
| **Individual household** | HDSS, community health | Exact GPS (rare in public datasets) | Precision exposure mapping; strict privacy controls required |

**Why does the choice of spatial unit matter?**

- A **facility record** only captures care-seekers, not the whole population — useful for access studies but cannot represent population birth rates.
- A **district aggregate** covers everyone but erases within-district environmental variation. A 5,000 km² district averages over steep temperature or rainfall gradients that matter for health.
- A **DHS cluster** gives individual birth records from a probability sample, enabling regression with individual-level confounders — but coordinates are displaced up to 5 km to protect participant privacy.

**In this course** we use DHS cluster data. The spatial join methods you learn in later notebooks apply equally to facility coordinates, ward centroids, or household GPS.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.cm import get_cmap

import pathlib, sys

def _find_data():
    here = pathlib.Path().resolve()
    for p in (here, here.parent, here.parent.parent):
        if (p / 'data').is_dir():
            return p / 'data'
    raise FileNotFoundError('Cannot find data/ directory')

DATA_DIR = _find_data()
print(f'Data directory: {DATA_DIR}')

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

def mc_question(question, options, correct, explanation):
    try:
        import ipywidgets as w
        from IPython.display import display, HTML
        header = w.HTML(f'<div style="font-size:15px;font-weight:600;margin-bottom:10px">🧠 {question}</div>')
        radio = w.RadioButtons(options=[f"{k})  {v}" for k, v in options.items()],
                               layout=w.Layout(width="100%"),
                               style={"description_width": "0px"})
        feedback = w.Output()
        def _check(change):
            if change["name"] != "value": return
            letter = change["new"][0]
            with feedback:
                feedback.clear_output(wait=True)
                if letter == correct:
                    display(HTML(f'<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;margin-top:6px;border-left:4px solid #28a745">✅ <b>Correct!</b>&nbsp; {explanation}</div>'))
                else:
                    display(HTML(f'<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;margin-top:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b>&nbsp; The answer is <b>{correct}</b>. {explanation}</div>'))
        radio.observe(_check, names="value")
        display(w.VBox([header, radio, feedback],
                       layout=w.Layout(border="1px solid #dee2e6", padding="16px 20px",
                                       border_radius="8px", margin="16px 0")))
    except ImportError:
        print(f"Q: {question}")
        for k, v in options.items():
            print(f"  {k}) {v}{' ← CORRECT' if k == correct else ''}")
        print(f"Explanation: {explanation}")


---
## 2. What Does a Spatial Health Record Look Like?

The Zimbabwe DHS 2015 birth dataset contains records for births in the five years prior to fieldwork (birth years 2011–2015), reported retrospectively by women aged 15–49. Each row in the dataset represents **one birth**.

A typical record contains four groups of attributes:

| Attribute group | Examples | Purpose in analysis |
|---|---|---|
| **Location** | `cluster_id`, `cluster_lat/lon`, `province`, `urban` | Defines *where* to link environmental exposures |
| **Time** | `birth_year`, `birth_month`, `conception_year` | Defines *when* — the trimester window for exposure assignment |
| **Health outcome** | `birth_weight_g`, `low_birth_weight`, `neonatal_death`, `preterm` | What we are trying to explain |
| **Covariates** | `mother_age`, `parity`, `wealth_index`, `anc_4plus`, `facility_delivery` | Confounders and effect modifiers |

> **Key design feature:** Sampling is two-stage — clusters (enumeration areas) are drawn first, then households within clusters. Records within the same cluster are *not* independent, and a `sample_weight` corrects for differential sampling probabilities.

The cell below loads the data and shows its dimensions, birth-year range, provinces covered, and the extent of missing values in the outcome columns — all important to understand before any analysis.


In [ ]:
births = pd.read_csv(DATA_DIR / 'zw_births_raw.csv')

print(f"Shape: {births.shape[0]:,} rows × {births.shape[1]} columns")
print(f"Birth years: {births['birth_year'].min()}–{births['birth_year'].max()}")
print(f"Provinces ({births['province'].nunique()}): {sorted(births['province'].unique())}")
print()

# Missing data is an important feature of DHS — not every outcome is recorded for every birth
outcome_cols = ['birth_weight_g', 'low_birth_weight', 'neonatal_death',
                'small_at_birth', 'preterm', 'anc_4plus', 'facility_delivery']
missing = births[outcome_cols].isna().sum()
missing_pct = (missing / len(births) * 100).round(1)
print("Missing values in key outcome columns:")
print(pd.DataFrame({'n_missing': missing, 'pct_missing': missing_pct}).to_string())

> **Data quality note — two traps to avoid:**
>
> 1. **CMC date encoding.** DHS raw files record dates as *century month codes* (months since January 1900). The columns `birth_year`, `birth_month` etc. have already been decoded to calendar values in this dataset.
>
> 2. **Sentinel values in `birth_weight_g`.** Codes ≥ 9000 (e.g. 9996 = "not weighed", 9998 = "don't know") are *not* real weights. Any birth weight analysis must filter to `birth_weight_g < 9000`. Forgetting this inflates apparent mean birth weights in the rural subsample.


---
## 3. Why Location Matters: Low Birth Weight Across Provinces

Low birth weight (LBW, defined as < 2,500 g) is one of the strongest predictors of infant mortality and long-term developmental outcomes. If LBW were purely a biological or genetic phenomenon, we would expect roughly uniform rates across the country. Instead, DHS data consistently shows **geographic variation** — some provinces have rates nearly double others.

That spatial variation is the entry point for environmental health research: *what is different about these places?* Rainfall? Temperature? Air quality? Access to healthcare? This course is about building the analytical toolkit to answer exactly that question.

The chart below shows the LBW rate (among births with a real recorded weight) by province. Note that rural, home-delivered births are disproportionately *unweighed* — so the sample here is already positively selected. The true rural LBW burden is likely higher than these numbers suggest.


In [ ]:
# Exclude sentinel codes ≥ 9000 in addition to NA — these are "not weighed" placeholders
weighed = births[(births['birth_weight_g'].notna()) & (births['birth_weight_g'] < 9000)].copy()

lbw_by_province = (
    weighed.groupby('province')
    .agg(
        n_with_weight=('birth_weight_g', 'count'),
        n_lbw=('low_birth_weight', 'sum')
    )
    .assign(lbw_rate_pct=lambda d: (d['n_lbw'] / d['n_with_weight'] * 100).round(2))
    .reset_index()[['province', 'lbw_rate_pct', 'n_with_weight']]
    .sort_values('lbw_rate_pct', ascending=False)
    .reset_index(drop=True)
)

print(lbw_by_province.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(lbw_by_province['province'][::-1],
        lbw_by_province['lbw_rate_pct'][::-1],
        color='#c0392b', edgecolor='white', linewidth=0.5)
ax.axvline(lbw_by_province['lbw_rate_pct'].mean(),
           color='#2c3e50', linestyle='--', linewidth=1.2,
           label=f"Mean across provinces ({lbw_by_province['lbw_rate_pct'].mean():.1f}%)")
ax.set_xlabel('LBW rate (%)')
ax.set_title('Low Birth Weight Rate by Province\nZimbabwe DHS 2015 (births with real recorded weight)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">&#129504; The Zimbabwe DHS 2015 data shows that LBW rates vary substantially across provinces. What does this geographic clustering of LBW MOST suggest?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb01q1" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> LBW is mainly genetic and the provincial variation is just random sampling noise</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb01q1" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> Geographic variation in LBW reflects differences in local environment, health access, and socioeconomic conditions — exactly the factors this course investigates</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb01q1" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> Provinces with more births automatically show higher LBW rates</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb01q1" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> Provincial LBW differences disappear after adjusting for birth year</span></label></div>
  <div id="nb01q1-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="B",E="Geographic clustering of LBW is not random — it reflects the interplay of environmental exposures, health system access, and socioeconomic factors that vary spatially. Unpacking this is the core mission of the GeoAI pipeline.",ID="nb01q1";
  function showFeedback(val){
    var fb=document.getElementById(ID+"-fb");
    if(!fb)return;
    if(val===C){
      fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';
    }else{
      fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';
    }
    document.dispatchEvent(new CustomEvent('mcq-update'));
  }
  function attach(){
    var saved=localStorage.getItem('mcq_'+ID);
    document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){
      if(saved&&r.value===saved){r.checked=true;}
      r.addEventListener("change",function(){
        localStorage.setItem('mcq_'+ID,this.value);
        showFeedback(this.value);
      });
    });
    if(saved)showFeedback(saved);
  }
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

---
## 4. How DHS Encodes Location — The Cluster and the Jitter

Each birth record is linked to a **cluster** (DHS enumeration area). The cluster's GPS coordinates are the centroid of the enumeration area *plus a controlled random offset* — the DHS Programme calls this displacement, and it exists to protect respondent confidentiality:

| Stratum | Displacement |
|---------|-------------|
| Urban clusters | 0–2 km in a random direction |
| Rural clusters | 0–5 km (99% of clusters) |
| Rural clusters (1% random selection) | 0–10 km |

Displacement is constrained to stay within the same administrative district.

**What this means for environmental analysis:** When we extract satellite-derived temperature or rainfall at the displaced coordinate, we introduce *measurement error in the exposure*, not the outcome. This is non-differential misclassification — it attenuates (shrinks) the estimated association toward zero. Analyses using spatial buffers of 5+ km partially mitigate this bias by averaging over the displacement range.

The map below shows all 399 DHS clusters from the 2015 Zimbabwe survey, coloured by province.


In [ ]:
clusters = pd.read_csv(DATA_DIR / 'zw_clusters.csv')
print(f"Clusters: {len(clusters)}  |  Provinces: {clusters['province'].nunique()}")
print(f"Urban clusters: {(clusters['urban'] == 1).sum()}  |  Rural: {(clusters['urban'] == 0).sum()}")
clusters.head(3)

In [ ]:
import contextily as ctx
from pyproj import Transformer as _T

provinces = sorted(clusters['province'].unique())
cmap = get_cmap('tab10', len(provinces))
province_colour = {p: cmap(i) for i, p in enumerate(provinces)}

# Project to Web Mercator for basemap
_proj = _T.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
_cx, _cy = _proj.transform(clusters['cluster_lon'].values, clusters['cluster_lat'].values)
clusters = clusters.copy()
clusters['_mx'] = _cx; clusters['_my'] = _cy

fig, ax = plt.subplots(figsize=(9, 8))
for province in provinces:
    sub = clusters[clusters['province'] == province]
    ax.scatter(sub['_mx'], sub['_my'],
               c=[province_colour[province]], s=25, alpha=0.85, linewidths=0)

try:
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, zoom=7)
except Exception:
    pass

ax.set_axis_off()
legend_handles = [mpatches.Patch(color=province_colour[p], label=p) for p in provinces]
ax.legend(handles=legend_handles, fontsize=8, loc='lower left',
          title='Province', title_fontsize=9, framealpha=0.9)
ax.set_title('Zimbabwe DHS 2015 — Cluster Locations by Province\n399 clusters, coordinates randomly displaced 0–5 km',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Urban–Rural Gradients in Birth Outcomes

Zimbabwe's DHS design deliberately over-samples urban clusters to allow urban-specific estimates. This means the `urban` column (0 = rural, 1 = urban) is not just a covariate — it is also a **stratum label** that reflects different sampling intensities.

Urban and rural areas differ along many dimensions simultaneously: proximity to health facilities, household wealth, mother's education, road quality, and — crucially for this course — climate exposure. A rural cluster in Zimbabwe's lowveld experiences different temperature, rainfall, and NDVI than a peri-urban cluster in Harare. Disentangling the environmental contribution from the socioeconomic one is the analytical challenge at the heart of this course.

The tables below compare three indicators across urban and rural strata. These are **unweighted** descriptive statistics — for nationally representative figures, `sample_weight` should be applied, since urban clusters are over-sampled.


In [ ]:
nm_by_province = (
    births.groupby('province')
    .agg(
        n_births=('neonatal_death', 'count'),
        n_deaths=('neonatal_death', 'sum')
    )
    .assign(nm_rate=lambda d: (d['n_deaths'] / d['n_births'] * 1000).round(1))
    .sort_values('nm_rate', ascending=False)
)

highest_nm_province = str(nm_by_province.index[0])
highest_nm_rate = float(nm_by_province['nm_rate'].iloc[0])

print(nm_by_province[['n_births', 'n_deaths', 'nm_rate']].to_string())
print(f"\nHighest neonatal mortality: {highest_nm_province} ({highest_nm_rate} per 1,000 births)")

In [ ]:
stratum_labels = {0: 'Rural', 1: 'Urban'}

# Exclude sentinel codes before computing birth weight statistics
real_weight = births[(births['birth_weight_g'].notna()) & (births['birth_weight_g'] < 9000)]

bw_by_stratum = (
    real_weight
    .groupby('urban')['birth_weight_g']
    .agg(['mean', 'count'])
    .rename(index=stratum_labels)
    .rename(columns={'mean': 'mean_bw_g', 'count': 'n'})
    .round({'mean_bw_g': 1})
)

# Facility delivery rate
fd_by_stratum = (
    births.dropna(subset=['facility_delivery'])
    .groupby('urban')['facility_delivery']
    .agg(['mean', 'count'])
    .rename(index=stratum_labels)
    .rename(columns={'mean': 'facility_delivery_pct', 'count': 'n'})
    .assign(facility_delivery_pct=lambda d: (d['facility_delivery_pct'] * 100).round(1))
)

# ANC 4+ coverage
anc_by_stratum = (
    births.dropna(subset=['anc_4plus'])
    .groupby('urban')['anc_4plus']
    .agg(['mean', 'count'])
    .rename(index=stratum_labels)
    .rename(columns={'mean': 'anc4plus_pct', 'count': 'n'})
    .assign(anc4plus_pct=lambda d: (d['anc4plus_pct'] * 100).round(1))
)

print("Mean birth weight (g) — real recorded weights only (sentinel codes excluded):")
print(bw_by_stratum.to_string())
print()
print("Facility delivery rate (%):")
print(fd_by_stratum.to_string())
print()
print("ANC 4+ coverage (%):")
print(anc_by_stratum.to_string())

Urban areas show substantially higher facility delivery and ANC 4+ rates — consistent with better geographic access to services and higher average wealth and education.

The birth weight gap between urban and rural is smaller and should be interpreted carefully: rural births that *have* a recorded weight are disproportionately facility deliveries, so the rural birth weight sample is already positively selected relative to all rural births. This is a common source of selection bias in DHS analyses.


In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">&#129504; Urban births in Zimbabwe show higher facility delivery rates, yet the LBW rate gap between urban and rural is smaller than expected. What is the most plausible explanation?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb01q2" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> Rural mothers eat more natural foods which protects against LBW</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb01q2" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> The DHS systematically over-sampled wealthy urban women, inflating the urban advantage</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb01q2" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> Urban advantages in access and wealth may be partly offset by urban-specific exposures — air pollution, heat, crowding — illustrating why crude urban/rural labels are not enough</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb01q2" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> The sample is too small to draw any conclusions about urban-rural differences</span></label></div>
  <div id="nb01q2-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="C",E="The urban advantage in facility delivery does not translate cleanly into better birth outcomes because the urban environment itself differs — heat island effects, air pollution, and crowding can harm pregnancy. This is why the course measures actual exposures rather than using urban/rural as a proxy.",ID="nb01q2";
  function showFeedback(val){
    var fb=document.getElementById(ID+"-fb");
    if(!fb)return;
    if(val===C){
      fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';
    }else{
      fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';
    }
    document.dispatchEvent(new CustomEvent('mcq-update'));
  }
  function attach(){
    var saved=localStorage.getItem('mcq_'+ID);
    document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){
      if(saved&&r.value===saved){r.checked=true;}
      r.addEventListener("change",function(){
        localStorage.setItem('mcq_'+ID,this.value);
        showFeedback(this.value);
      });
    });
    if(saved)showFeedback(saved);
  }
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

---
## Recap

**What spatial health data looks like:**
Every record in a spatial health dataset needs at minimum: a location (coordinates or area code), a time, and an outcome. DHS birth records also carry rich individual covariates — enabling regression analyses that go beyond aggregate comparisons.

**Key data quality points to carry forward:**
- **Missing values are structured, not random.** Birth weight is missing far more often in rural, home-delivered births — making the observed sample positively selected.
- **Coordinate displacement** introduces non-differential exposure measurement error that attenuates associations. Buffers ≥ 5 km partially mitigate this.
- **Sample weights** are required for nationally representative estimates. Urban over-sampling biases unweighted national statistics toward urban outcomes.
- **Urban–rural gradients** are strong in Zimbabwe DHS data: urban women have markedly higher facility delivery and ANC coverage rates. These access differences are major confounders in exposure–outcome analyses.

**In Notebook 02** we explore the satellite and reanalysis climate data that provides the exposure columns (`temp_t1/t2/t3`, `rain_t1/t2/t3`, `ndvi_t1/t2/t3`, `pm25_t1/t2/t3`). Understanding Zimbabwe's seasonal structure is the necessary foundation before interpreting those exposures.


In [ ]:
%%html
<div id="nb01-scoreboard" style="font-family:system-ui,sans-serif;border:2px solid #adb5bd;border-radius:10px;padding:20px 24px;margin:24px 0;background:#f8f9fa">
  <div style="font-size:17px;font-weight:700;margin-bottom:12px;color:#1a1a2e">&#128202; Module 1 — Spatial Health Data &#8212; Quiz Results</div>
  <div id="nb01-score-display"><p style="color:#856404;background:#fff3cd;padding:10px 12px;border-radius:6px;margin:0">&#9203; Complete all reflection questions above to see your score.</p></div>
  <div id="nb01-next-action" style="margin-top:14px"></div>
</div>
<script>
(function(){
  var QUESTIONS=[{id:"nb01q1",correct:"B"},{id:"nb01q2",correct:"C"}];
  var PASS=50;
  var NEXT="02_exposure_extraction_linkage.ipynb";
  var NEXT_LABEL="Notebook 02: Exposure Extraction and Linkage";
  var NBID="nb01";
  function update(){
    var correct=0,answered=0;
    QUESTIONS.forEach(function(q){
      var ans=localStorage.getItem("mcq_"+q.id);
      if(ans!==null){answered++;if(ans===q.correct)correct++;}
    });
    var total=QUESTIONS.length;
    var pct=total?Math.round(correct/total*100):0;
    var passed=pct>=PASS;
    var disp=document.getElementById(NBID+"-score-display");
    var act=document.getElementById(NBID+"-next-action");
    if(!disp)return;
    if(answered<total){
      disp.innerHTML='<p style="color:#856404;background:#fff3cd;padding:10px 12px;border-radius:6px;margin:0">&#9203; Answer all '+total+' questions to see your score. ('+( total-answered)+' remaining)</p>';
      if(act)act.innerHTML="";
    }else if(passed){
      disp.innerHTML='<div style="background:#d4edda;color:#155724;padding:12px 16px;border-radius:8px"><b>&#10003; Score: '+correct+'/'+total+' ('+pct+'%) &#8212; MODULE PASSED</b><br><small>Scroll down and run the <b>Save Progress</b> cell to unlock the next notebook.</small></div>';
      if(act){
        localStorage.setItem(NBID+"_passed","true");
        if(NEXT){act.innerHTML='<div style="margin-top:10px"><a href="'+NEXT+'" style="display:inline-block;background:#198754;color:#fff;padding:10px 22px;border-radius:6px;text-decoration:none;font-weight:600;font-size:14px">&#10145; Continue to '+NEXT_LABEL+'</a><div style="margin-top:8px;font-size:12px;color:#6c757d">&#128269; Also run the <b>Save Progress</b> cell below — it unlocks the prereq gate in the next notebook.</div></div>';}
      }
    }else{
      disp.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:12px 16px;border-radius:8px"><b>&#10007; Score: '+correct+'/'+total+' ('+pct+'%) &#8212; Below pass mark ('+PASS+'%)</b><br><small>Go back to any question and change your answer.</small></div>';
      if(act)act.innerHTML="";
    }
  }
  document.addEventListener("mcq-update",update);
  if(document.readyState==="loading")document.addEventListener("DOMContentLoaded",update);
  else update();
  setInterval(update,3000);
})();
</script>

In [ ]:
import pathlib as _pl
# ── Save progress — run this after passing the quiz above ─────────────────
_badge = _pl.Path.home() / '.nb01_complete'
_badge.touch()
print(f"✅  Progress saved: {_badge}")
print("   You can now open Notebook 02: Exposure Extraction and Linkage.")

---

## ✅ Notebook 01 complete

You have a clean, geocoded dataset of Zimbabwe DHS birth records — each with a location, a birth outcome, and mother characteristics.

**What is missing?** The environment. But before we explore the climate data, it is worth asking: where does that data actually come from?

➡️ **Notebook 02** introduces Google Earth Engine and the satellite products used to characterise Zimbabwe's climate — before we look at the data itself, we understand how it was made.